In [80]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/"
OUTPUT_DIR = "../data/processed/"

employment_path = DATA_DIR + "就业信息.xlsx"

unemployment = pd.read_excel(
    employment_path,
    sheet_name="城镇失业率"
)

employees = pd.read_excel(
    employment_path,
    sheet_name="从业人员数"
)

industry = pd.read_excel(
    employment_path,
    sheet_name="第一、二、三产业就业人数"
)


# print("城镇失业率：", unemployment.shape)
# print("从业人员数：", employees.shape)
# print("三次产业就业人数：", industry.shape)



In [81]:
unemployment.head()

,年份,城市名称,unemploymentRate
0,2001,city1,3.0
1,2002,city1,4.0
2,2003,city1,4.0
3,2005,city1,4.0
4,2006,city1,4.0


In [82]:
employees.head()

,城市名称,年份,employeesNumber
0,city1,2021,199
1,city1,2019,194
2,city1,2018,192
3,city1,2017,189
4,city1,2016,187


In [83]:
industry.head()

,城市名称,年份,pi_Employment,si_Employment,ti_Employment
0,city1,2021,0,101,89
1,city1,2019,0,95,79
2,city1,2018,0,96,74
3,city1,2017,0,96,72
4,city1,2016,0,96,69


统一字段名称

In [84]:
unemployment = unemployment.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "unemploymentRate": "unemployment_rate"
})

employees = employees.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "employeesNumber": "employees_number"
})

industry = industry.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "pi_Employment": "primary_employment",
    "si_Employment": "secondary_employment",
    "ti_Employment": "tertiary_employment"
})


for df in [unemployment, employees, industry]:
    df["city_id"] = (
        df["city_id"]
        .astype(str)
        .str.strip()
    )

    df["year"] = pd.to_numeric(
        df["year"],
        errors="coerce"
    )

检查城市覆盖情况

In [85]:
for name, df in {
    "失业率": unemployment,
    "从业人员": employees,
    "产业就业": industry
}.items():

    print("=" * 50)
    print(name)
    print("城市数：", df["city_id"].nunique())
    print("年份：", df["year"].min(), "-", df["year"].max())

失业率
城市数： 38
年份： 2000 - 2022
从业人员
城市数： 35
年份： 2000 - 2021
产业就业
城市数： 40
年份： 2002 - 2021


寻找重复记录

In [86]:
employees_dup = employees[
    employees.duplicated(
        ["city_id", "year"],
        keep=False
    )
].sort_values(["city_id", "year"])

employees_dup

,city_id,year,employees_number
485,city25,2004,386
503,city25,2004,260
484,city25,2005,396
502,city25,2005,275
483,city25,2006,415
...,...,...,...
664,city33,2018,446
645,city33,2019,576
663,city33,2019,432
644,city33,2020,576


一致性检查

In [87]:
industry["industry_employment_total"] = (
    industry["primary_employment"]
    + industry["secondary_employment"]
    + industry["tertiary_employment"]
)

industry.head()

,city_id,year,primary_employment,secondary_employment,tertiary_employment,industry_employment_total
0,city1,2021,0,101,89,190
1,city1,2019,0,95,79,174
2,city1,2018,0,96,74,170
3,city1,2017,0,96,72,168
4,city1,2016,0,96,69,165


In [88]:
duplicate_check = employees_dup.merge(
    industry[
        [
            "city_id",
            "year",
            "industry_employment_total"
        ]
    ],
    on=["city_id", "year"],
    how="left"
)

duplicate_check["difference"] = abs(
    duplicate_check["employees_number"]
    - duplicate_check["industry_employment_total"]
)

# duplicate_check.head(30)

异常统计表，这边可以解释删除什么异常

In [89]:
duplicate_check.to_csv(
    OUTPUT_DIR + "employees_duplicate_audit.csv",
    index=False
)

In [90]:
employees = (
    employees
    .sort_values(
        ["city_id", "year", "employees_number"],
        ascending=[True, True, False]
    )
    .drop_duplicates(
        ["city_id", "year"],
        keep="first"
    )
    .reset_index(drop=True)
)



# employees.duplicated(
#     ["city_id", "year"]
# ).sum()


三张表统一排序

In [91]:
unemployment = (
    unemployment
    .sort_values(["city_id", "year"])
    .reset_index(drop=True)
)

employees = (
    employees
    .sort_values(["city_id", "year"])
    .reset_index(drop=True)
)

industry = (
    industry
    .sort_values(["city_id", "year"])
    .reset_index(drop=True)
)

质量报告

In [92]:
def quality_report(df, name):
    print("=" * 60)
    print(name)

    print("Shape：", df.shape)
    print("城市数量：", df["city_id"].nunique())
    print("年份范围：", df["year"].min(), "-", df["year"].max())

    print(
        "重复 city-year：",
        df.duplicated(["city_id", "year"]).sum()
    )

    print("缺失值：")
    print(df.isna().sum())
    
    
quality_report(
    unemployment,
    "城镇失业率"
)

quality_report(
    employees,
    "从业人员数"
)

quality_report(
    industry,
    "三次产业就业人数"
)

城镇失业率
Shape： (769, 3)
城市数量： 38
年份范围： 2000 - 2022
重复 city-year： 0
缺失值：
year                 0
city_id              0
unemployment_rate    1
dtype: int64
从业人员数
Shape： (690, 3)
城市数量： 35
年份范围： 2000 - 2021
重复 city-year： 0
缺失值：
city_id             0
year                0
employees_number    0
dtype: int64
三次产业就业人数
Shape： (698, 6)
城市数量： 40
年份范围： 2002 - 2021
重复 city-year： 0
缺失值：
city_id                      0
year                         0
primary_employment           0
secondary_employment         0
tertiary_employment          0
industry_employment_total    0
dtype: int64


保存结果

In [93]:
unemployment.to_csv(
    OUTPUT_DIR + "unemployment_clean.csv",
    index=False
)

employees.to_csv(
    OUTPUT_DIR + "employees_clean.csv",
    index=False
)

industry.to_csv(
    OUTPUT_DIR + "industry_employment_clean.csv",
    index=False
)